# E20 — O fecho: a promessa com os buracos abertos ao mesmo tempo

Dezenove capítulos mediram buracos separados: a memória que cega, o atraso do dado, o pedaço
escolhido, a mudança que chega sem avisar, a cauda que vem junto, o passado que não volta, o
trabalho que se gasta. Cada um tem o seu preço medido, e nenhum deles foi medido junto com os
outros.

Este caderno mede a promessa do primeiro capítulo com **três buracos abertos ao mesmo tempo**: a
janela da calibração (curta, média ou longa), o atraso do dado (nenhum ou vinte e um dias) e o mundo
(parado ou dobrando no meio). São doze configurações, e a promessa é a mesma em todas: cinco por
cento dos dias rompendo o corte, com o posto acompanhando a janela para que a taxa anunciada não
mude.

A pergunta é se os buracos somam ou multiplicam.


In [1]:
# <- brinque com: JANELAS, ATRASO, SEMENTES, DIAS, QUANDO, FATOR, HORIZONTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import graficos, mudanca, promessa

JANELAS = (126, 252, 504)
ATRASO = 21
ATRASADOS = (0, ATRASO)
FATORES = (1.0, 2.0)
SEMENTES = 20
DIAS = 12000
QUANDO = 6000
HORIZONTE = 250
CAUDA = 0.05
SEMENTE = 800

print("a promessa anunciada e de %.4f em todas as configuracoes" % CAUDA)
for janela in JANELAS:
    print("  janela %3d dias | posto %2d | a entrega que o corte assina sozinho: %.5f"
          % (janela, promessa.posto(janela, CAUDA), promessa.entrega_do_corte(janela, CAUDA)))


a promessa anunciada e de 0.0500 em todas as configuracoes
  janela 126 dias | posto  7 | a entrega que o corte assina sozinho: 0.05512
  janela 252 dias | posto 13 | a entrega que o corte assina sozinho: 0.05138
  janela 504 dias | posto 26 | a entrega que o corte assina sozinho: 0.05149


In [2]:
# As doze configuracoes: o mundo, a janela e o atraso.
linhas = []
for fator in FATORES:
    for janela in JANELAS:
        for atraso in ATRASADOS:
            valores = []
            for i in range(SEMENTES):
                x = mudanca.degrau(DIAS, np.random.default_rng(SEMENTE + i), fator=fator, quando=QUANDO)
                s = pd.Series(x, index=pd.RangeIndex(DIAS))
                corte = promessa.corte_no_posto(s, janela, promessa.posto(janela, CAUDA)).shift(atraso)
                rompe = (s < corte).loc[QUANDO:QUANDO + HORIZONTE]
                valores.append(float(rompe.mean(skipna=True)))
            linhas.append({"mundo": "mudado" if fator > 1.0 else "parado", "janela": janela,
                           "atraso": atraso, "entrega": float(np.mean(valores)),
                           "vezes": float(np.mean(valores) / CAUDA),
                           "dispersao": float(np.std(valores, ddof=1))})
fecho = pd.DataFrame(linhas).set_index(["mundo", "janela", "atraso"])
print(fecho.round(4).to_string())


                      entrega   vezes  dispersao
mundo  janela atraso                            
parado 126    0        0.0566  1.1315     0.0083
              21       0.0558  1.1155     0.0109
       252    0        0.0526  1.0518     0.0100
              21       0.0518  1.0359     0.0114
       504    0        0.0508  1.0159     0.0137
              21       0.0506  1.0120     0.0146
mudado 126    0        0.0849  1.6972     0.0079
              21       0.0954  1.9084     0.0105
       252    0        0.1050  2.0996     0.0116
              21       0.1213  2.4263     0.0122
       504    0        0.1502  3.0040     0.0151
              21       0.1602  3.2032     0.0177


In [3]:
# O que cada buraco faz sozinho, e o que os tres fazem juntos.
parado = fecho.loc["parado"]
mudado = fecho.loc["mudado"]
so_janela = (parado.loc[(504, 0), "vezes"], mudado.loc[(504, 0), "vezes"])
so_atraso = (parado.loc[(252, 0), "vezes"], parado.loc[(252, ATRASO), "vezes"])
so_mudanca = (parado.loc[(252, 0), "vezes"], mudado.loc[(252, 0), "vezes"])
tudo = mudado.loc[(504, ATRASO), "vezes"]
print("janela longa: no mundo parado %.2f, no mundo que dobra %.2f" % so_janela)
print("atraso de %d dias no mundo parado: %.2f sem ele, %.2f com ele" % (ATRASO, so_atraso[0], so_atraso[1]))
print("mudanca com a janela media e sem atraso: %.2f" % so_mudanca[1])
print()
print("os tres juntos: %.2f vezes a promessa (janela de 504, atraso de %d, mundo que dobra)"
      % (tudo, ATRASO))
print("a soma dos efeitos isolados seria de %.2f vezes" % (1.0 + (so_janela[1] - so_janela[0]) + (so_atraso[1] - so_atraso[0])))


janela longa: no mundo parado 1.02, no mundo que dobra 3.00
atraso de 21 dias no mundo parado: 1.05 sem ele, 1.04 com ele
mudanca com a janela media e sem atraso: 2.10

os tres juntos: 3.20 vezes a promessa (janela de 504, atraso de 21, mundo que dobra)
a soma dos efeitos isolados seria de 2.97 vezes


In [4]:
# Figura 1: a entrega contra a janela, nos dois mundos, sem atraso.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
for mundo, cor in (("parado", "#1f4e79"), ("mudado", "#b03a2e")):
    valores = [fecho.loc[(mundo, j, 0), "vezes"] for j in JANELAS]
    eixo.plot(JANELAS, valores, marker="o", color=cor, lw=1.8,
              label="mundo %s, sem atraso" % mundo)
    valores_atraso = [fecho.loc[(mundo, j, ATRASO), "vezes"] for j in JANELAS]
    eixo.plot(JANELAS, valores_atraso, marker="s", ls="--", color=cor, lw=1.4,
              label="mundo %s, com %d dias de atraso" % (mundo, ATRASO))
eixo.axhline(1.0, color="#555555", ls=":", lw=1.4, label="a promessa")
eixo.set_xticks(list(JANELAS))
eixo.set_xlabel("janela da calibração, em dias")
eixo.set_ylabel("entrega, em vezes a promessa")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25)
fig.tight_layout()
graficos.salvar(fig, "E20_o_fecho", 1)
plt.close(fig)
print("no mundo parado as vezes vao de %.2f a %.2f; no mudado, de %.2f a %.2f"
      % (fecho.loc[("parado", 126, 0), "vezes"], fecho.loc[("parado", 504, 0), "vezes"],
         fecho.loc[("mudado", 126, 0), "vezes"], fecho.loc[("mudado", 504, ATRASO), "vezes"]))


no mundo parado as vezes vao de 1.13 a 1.02; no mudado, de 1.70 a 3.20


In [5]:
# Figura 2: o preco dos buracos, um a um e todos juntos.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
rotulos = ["a promessa", "so a janela\nlonga", "so o atraso\nde %d dias" % ATRASO,
           "so a mudança", "janela e\nmudança", "a pilha\ninteira"]
valores = [1.0, so_janela[0], so_atraso[1], so_mudanca[1], so_janela[1], tudo]
cores = ["#555555", "#1f4e79", "#1f4e79", "#b03a2e", "#b03a2e", "#2e7d32"]
eixo.bar(np.arange(len(valores)), valores, 0.6, color=cores)
for i, v in enumerate(valores):
    eixo.annotate("%.2f" % v, (i, v), textcoords="offset points", xytext=(0, 4), ha="center", fontsize=9)
eixo.set_xticks(np.arange(len(valores)))
eixo.set_xticklabels(rotulos, fontsize=8)
eixo.set_ylabel("entrega, em vezes a promessa")
eixo.grid(alpha=0.25, axis="y")
fig.tight_layout()
graficos.salvar(fig, "E20_o_fecho", 2)
plt.close(fig)
print("a pilha inteira entrega %.2f vezes a promessa" % tudo)


a pilha inteira entrega 3.20 vezes a promessa


## Leitura visual das figuras

**Figura 1 (quatro curvas de entrega contra a janela).** O eixo horizontal são as três janelas de calibração, 126, 252 e 504 dias; o vertical, a entrega em vezes a promessa. As duas curvas do mundo parado (azul) descem e praticamente se sobrepõem: com e sem os 21 dias de atraso a diferença visual é da espessura do traço, e as duas terminam encostadas em 1,01. As duas do mundo mudado (vermelho) sobem, e ali a linha tracejada (com atraso) fica **acima** da contínua (sem atraso) em todo o trecho — a distância entre elas cresce com a janela. O eixo engana num ponto: a escala é a mesma para os quatro casos, mas o azul e o vermelho se leem como se fossem o mesmo objeto, e não são — a janela longa é boa no mundo parado e péssima no mundo mudado, com o mesmo sinal de inclinação ao contrário. Nada se cruza: o que muda de mundo para mundo é o sentido da inclinação.

**Figura 2 (o preço dos buracos).** Seis barras, todas medidas na mesma régua, com a linha da promessa ao fundo. As três primeiras quase não se levantam do chão (a promessa, só a janela longa, só o atraso de 21 dias). A quarta, que é só a mudança, já passa de duas vezes. A quinta, janela e mudança juntas, sobe a três. A sexta, a pilha inteira, é visivelmente a mais alta das seis — e mais alta do que a soma do que as barras do meio acrescentam sozinhas. As duas últimas são quase da mesma altura, e é a leitura mais fina do gráfico: os buracos não se somam, eles se encaixam.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES_JANELA = {126: "curta", 252: "média", 504: "longa"}
NOMES_ATRASO = {0: "sem_atraso", ATRASO: "com_atraso"}
resultado = {
    "fecho_promessa": float(CAUDA),
    "fecho_janelas": int(len(JANELAS)),
    "fecho_atraso": int(ATRASO),
    "fecho_sementes": int(SEMENTES),
    "fecho_dias": int(DIAS),
    "fecho_horizonte": int(HORIZONTE),
    "fecho_configuracoes": int(len(fecho)),
    "fecho_janela_curta": int(JANELAS[0]),
    "fecho_janela_media": int(JANELAS[1]),
    "fecho_janela_longa": int(JANELAS[2]),
    "fecho_so_janela_parado": float(so_janela[0]),
    "fecho_so_janela_mudado": float(so_janela[1]),
    "fecho_so_atraso_parado": float(so_atraso[1]),
    "fecho_so_mudanca": float(so_mudanca[1]),
    "fecho_pilha": float(tudo),
    "fecho_soma_isolada": float(1.0 + (so_janela[1] - so_janela[0]) + (so_atraso[1] - so_atraso[0])),
}
for mundo in ("parado", "mudado"):
    for janela in JANELAS:
        for atraso in ATRASADOS:
            resultado["fecho_%s_%s_%s" % (mundo, NOMES_JANELA[janela], NOMES_ATRASO[atraso])] = float(
                fecho.loc[(mundo, janela, atraso), "vezes"])
            resultado["fecho_entrega_%s_%s_%s" % (mundo, NOMES_JANELA[janela], NOMES_ATRASO[atraso])] = float(
                fecho.loc[(mundo, janela, atraso), "entrega"])

caminho = Path("lab/resultados/E20_o_fecho.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E20_o_fecho.json gravado | 40 grandezas
